# **Artificial Intelligence — Lab Project**
## **From SLM to RAG to Graph RAG: Building Smarter Query Systems**

**Names:**   Αικατερίνα Γεωργάνα
**Student IDs:**  ge20097
**Team ID:** Ομάδα 28

This project explores how we can progressively enhance the query-answering capabilities of a Small Language Model over the script of the movie [12 Angry Men (1957)](https://www.imdb.com/title/tt0050083/), written by Reginald Rose — moving from raw SLM inference, to Retrieval-Augmented Generation (RAG), to Graph RAG with structured relational reasoning.

##**Introduction: From Simple LLM Inference to RAG and Graph RAG**

When designing AI systems that provide information, it is important to distinguish between three progressively more powerful paradigms:

### 1. Simple LLM / SLM Inference (No External Context)

A **Small Language Model (SLM)** or standard LLM used alone answers questions **only based on its internal parameters** (its training data and general knowledge). It does *not* access your script.

Characteristics:

* Works well for general knowledge.
* Cannot reliably answer questions about specific documents it has not seen.
* May hallucinate details about recent or specific events.

Example:

> “What is a large language model?”
> An SLM can answer this because it is general knowledge.

However:

> “What did Juror #10 say in Page 88?”
> An SLM alone cannot answer this because it has no access to that script.

---

### 2. Traditional RAG (Retrieval-Augmented Generation)

**RAG** enhances an LLM by retrieving relevant documents (or chunks) from a dataset and injecting them into the prompt as context.

Pipeline:

1. User question
2. Retriever finds relevant pages
3. LLM generates answer based on retrieved text

Characteristics:

* Excellent for **fact-based, context-dependent questions**.
* Works well when the answer is explicitly stated in one or a few documents.
* Still limited in **multi-step reasoning** across many documents.

Example:

> “According to Page 4, which jurors sit in the rear row?”
> RAG works well because the answer is explicitly in the script.

But:

> "Which juror grew up in a slum, and how does that connect to the knife testimony?"
> This requires structured reasoning across entities — something traditional RAG struggles with.

---

### 3. Graph RAG (Retrieval + Structured Reasoning)

**Graph RAG** extends RAG by:

* Extracting **entities** (jurors, witnesses, physical evidence, testimonies, locations)
* Modeling **relationships** between them (testified about, voted with, contradicted by, connected to, grew up in)
* Representing them as a **knowledge graph**
* Enabling multi-hop reasoning over structured connections

This enables:

* Multi-document reasoning
* Entity relationship tracing
* Pattern detection
* Causal inference
* Aggregation across documents

Graph RAG is particularly powerful when:

* The answer is not explicitly stated anywhere.
* The answer requires combining information from multiple pages.
* The reasoning involves relationships (e.g., shared backgrounds, testimony contradictions, behavioral patterns, evidence connections).

---

## **Part 1 — Understanding the Data and the Model**

Before building any system, you need to understand what you are working with.

In this part, you will explore the script of 12 Angry Men and interact directly with the raw SLM — no retrieval, no graph, no augmentation. This will help you develop an intuition for where the model succeeds on its own and, more importantly, where it fails.

From that understanding, you will design 15 questions that will serve as the evaluation backbone for the entire project:
- 5 questions answerable by the SLM alone (general knowledge, no script context needed)
- 5 questions that require retrieving specific information from one or a few pages (Traditional RAG territory)
- 5 questions that require reasoning across multiple pages and entities (Graph RAG territory)

**The goal is not to trick the model — it is to construct a dataset that clearly exposes the strengths and limits of each approach.**

This dataset is your Part 1 deliverable. It will be collected, merged with all submissions, and used as the shared evaluation dataset for Parts 2 and 3.

---

## Why These Categories Matter

### SLM-only questions

* Do not depend on dataset
* Test baseline knowledge
* No retrieval required

### Traditional RAG questions

* Answer explicitly exists in one or more pages
* Retrieval is sufficient
* No cross-document reasoning required

### Reasoning / Graph RAG questions

* Require:

  * Entity extraction
  * Relationship modeling
  * Multi-hop reasoning
  * Aggregation
* Cannot be solved by:

  * SLM alone (no context)
  * Naive RAG (no structured reasoning)

---

Your CSV must be createad with the **following structure**:

## Sample CSV

| Question                                                                                        | Category            | Correct Answer                       | Page References | Explanation                                                                         |
| ----------------------------------------------------------------------------------------------- | ------------------- | ------------------------------------ | ------------------ | ----------------------------------------------------------------------------------- |
| Who directed the movie 12 Angry Men (1957)?                                                                       | SLM only            | Sidney Lumet.             |                    | Movie titles and their directors are general knowledge and do not require context.      |
| How old is Juror #2?                   | Traditional RAG     | 38 years old.                        | 1                  | The SLM does not know this information. The retrieval of Page 1 is required. |
| Which juror's personal background made him the most credible witness to explain how a switchblade knife is properly used? | Reasoning/Graph RAG | Juror #5. | 31;121;122            |Requires entity extraction and multi-hop relationship reasoning. Page 31 mentions Juror #5 saying "I've lived in a slum all my life". Page 121 mentions Juror #5 demonstrating the underhanded switchblade technique, saying "Anyone who's ever used a switch-knife'd never handle it any other way". Page 122 mentions Juror #5 concluding the boy would not have made an overhand stab wound given his experience with switchblades.                |

---

#### 🔹 Step 1: Install dependencies *(these dependencies are for the whole project)*

In [2]:
!pip install llama-index graspologic llama-index-llms-huggingface llama-index-embeddings-huggingface transformers accelerate torch bitsandbytes pymupdf

#### 🔹 Step 2: Load the PDF and Create Documents

In this step we:

1. **Download** the PDF directly from its URL using `requests`.
2. **Extract** the text page-by-page using `pymupdf`.
3. **Convert** each page into a `Document` object — the standard input format for LlamaIndex.

---

##### What is a `Document`?

A `Document` in LlamaIndex is a lightweight container that wraps a piece of text together with optional metadata.  
You can think of it as a structured unit (e.g., a page or section) that the system will later split, embed, and index.

---

##### How We Structure the PDF

Instead of merging the entire PDF into one long string, we treat:

Each PDF page → one `Document`

This preserves:

- Page boundaries  
- Structural clarity  
- Easier traceability during retrieval  

Each document is created from the cleaned page text (after removing the printed page number such as `1.`, `2.`, etc.) and can optionally include metadata like:

```python
{
  "page_number": i,
  "source": "<pdf_url>"
}

In [3]:
import requests
import fitz  # pymupdf
from llama_index.core import Document

url = "https://f004.backblazeb2.com/file/screenplays/posts/12-angry-men-1957/scripts/12%20Angry%20Men%20-%20Release.pdf"

# Download PDF
response = requests.get(url)
response.raise_for_status()

# Open PDF from memory
doc = fitz.open(stream=response.content, filetype="pdf")

documents = []

for i, page in enumerate(doc, start=1):
    text = page.get_text()

    # Remove leading page number like "1.", "2.", etc.
    lines = text.splitlines()
    if lines and lines[0].strip().startswith(f"{i}."):
        lines = lines[1:]

    cleaned_text = "\n".join(lines).strip()

    # Create LlamaIndex Document
    document = Document(
        text=cleaned_text,
        metadata={
            "page_number": i,
            "source": url,
            "title": "12 Angry Men"
        }
    )

    documents.append(document)


#### 🔹 Step 3: Exploring the Dataset

Before designing your questions, take time to read some of the script's pages.
Use the cell below to browse through them — change the index to access any page.

In [4]:
# Change this index to browse different pages
page_index = 70   # 0 = first page, 1 = second page, etc.
print(f"Page Number: {documents[page_index].metadata['page_number']}")
print("-" * 80)
print(documents[page_index].text)

Page Number: 71
--------------------------------------------------------------------------------
#6
A guy who talks like that old man
oughta really get stepped on y'know.
#3
Get your hands off me!
#6
You oughta have some respect,
mister. If you say stuff like that
to him again -- I'm gonna lay you
out.
(he releases the #3 and 
speaks to #9)
Go ahead. You can say anything you
want. Why do you think the old man
might lie?
#9
It's just that I looked at him for a
very long time. The seam of his
jacket was split under his arm. Did
you notice it? I mean, to come into
court like that. He was a very old
man with a torn jacket and he walked
very slowly to the stand. He was
dragging his left leg and trying to
hide it because he was ashamed. I
think I know him better than anyone
her. This is a quiet, frightened,
insignificant old man who has been
nothing all his life, who has never
had recognition, his name in the
newspapers. Nobody knows him, nobody
quotes him, nobody seeks his advice
after seve

#### 🔹 Step 4: Load the SLM from HuggingFace

This is the most important setup step. We load the AI model locally.

### 🤖 Language Model (SLM): `Qwen/Qwen3-0.6B`
The LLM is responsible for **reading text and extracting entities and relationships** from it. We use Qwen3-0.6B, a compact but capable open-source model by Alibaba Cloud (~1.2GB).

Key parameters:
- `context_window=4096` — how many tokens the model can "see" at once.
- `max_new_tokens=512` — the maximum length of the model's response.
- `generate_kwargs` — controls randomness: `temperature=0.7` makes outputs creative but not chaotic.
- `device_map="auto"` — automatically places the model on GPU if available, otherwise CPU.

In [5]:
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen3-0.6B"


# ── 4-bit quantization config ────────────────────────────────────────────────
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Qwen3 thinking-mode: ALWAYS disable for structured extraction ─────────────
def messages_to_prompt(messages):
    """Apply Qwen3 chat template with thinking disabled."""
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs={"enable_thinking": True},
    )

def completion_to_prompt(completion):
    """Wrap a plain string prompt with thinking disabled."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": completion}],
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs={"enable_thinking": True},
    )

llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,
    context_window=4096,
    max_new_tokens=4096,
    generate_kwargs={"temperature": 0.1, "do_sample": False},  # deterministic
    model_kwargs={"quantization_config": quantization_config},
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto",
)



Settings.llm = llm


print(f"✅ LLM loaded: {MODEL_NAME} (4-bit quantized, thinking=OFF, temp=0.1)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ LLM loaded: Qwen/Qwen3-0.6B (4-bit quantized, thinking=OFF, temp=0.1)


#### 🔹 Step 5: Interacting with the SLM

Now test the model directly — no pages, no retrieval, just the raw SLM answering from its own knowledge.
Try asking general questions, then try asking something specific about the script.
Observe where it confidently answers and where it falls short or halluccinates.

In [6]:
# Change this to ask anything you want
your_question = "Which language is most widely known?"

response = llm.complete(your_question)

print("SLM Answer:")
print("=" * 60)
print(response.text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SLM Answer:
<think>
Okay, the user is asking which language is most widely known. I need to explain that the answer is English. But I should make sure to mention that the number one is English, and that other languages like Spanish, French, and others have similar popularity. I should also include some statistics to support the fact that English is the most widely spoken. Maybe add a brief note about the global reach of English. Let me check the facts again to confirm. Yes, English is the most widely spoken language, with over 300 million speakers worldwide. So the answer is English.
</think>

The language most widely known is **English**. It is the most commonly spoken language in the world, with over 300 million speakers globally. While other languages like Spanish, French, and many others have significant populations, English holds the largest number of speakers. This makes English the most widely known language.


#### 🔹 Step 6: Building Your Evaluation Dataset

Now that you have explored the pages and tested the SLM, it is time to design your 15 questions.
Be precise with your answers and references — this CSV will be your Part 1 deliverable and will form part of the shared evaluation dataset used in Parts 2 and 3. **The first column must be dedicated to your team's ID.**

> ⚠️ **Important:** For your 5 SLM-only questions, make sure the model actually answers them correctly before adding them to the dataset. Use the cell above to test each one. **If the SLM fails to answer a question reliably, it does not belong in that category**.

In [7]:
import pandas as pd

team_id = 28  # put your team's id here

questions = [
    # SLM only (5 questions)
    {"Question": "How many days does a week have?",        "Category": "SLM only",            "Correct Answer": "A week has 7 days. This is a standard time period, including weekends and weekdays. The number is fixed at 7.",                                  "Page References": "", "Explanation": "It is general knowledge and doesn't require context"},
    {"Question": "What is the chemical symbol for water?", "Category": "SLM only",            "Correct Answer": "The chemical symbol for water is **H₂O**. This represents a molecule consisting of two hydrogen atoms and one oxygen atom.",                     "Page References": "", "Explanation": "It is general knowledge and doesn't require context"},
    {"Question": "Which city is the capital of Greece?",   "Category": "SLM only",            "Correct Answer": "The capital of Greece is **Athens**.",                                                                                                            "Page References": "", "Explanation": "It is general knowledge and doesn't require context"},
    {"Question": "Who painted the Mona Lisa?",             "Category": "SLM only",            "Correct Answer": "The Mona Lisa is a famous painting by **Leonardo da Vinci**. ",                                                                                  "Page References": "", "Explanation": "It is general knowledge and doesn't require context"},
    {"Question": "Which language is most widely known?",   "Category": "SLM only",            "Correct Answer": "The language most widely known is **English**. It is the most commonly spoken language in the world, with over 300 million speakers globally.",  "Page References": "", "Explanation": "It is general knowledge and doesn't require context"},

    # Traditional RAG (5 questions)
    {"Question": "How many kids does Juror #8 have?",                   "Category": "Traditional RAG",     "Correct Answer": "He has three kids.",                                           "Page References": "30",      "Explanation": "The SLM does not know this information. The retrieval of Page 30 is required. In page 30 is mentioned that Juror #8 says 'three' after Juror #3 askes him ' Have you got any kids?' "},
    {"Question": "Which Juror is retired?",                             "Category": "Traditional RAG",     "Correct Answer": "Juror #9.",                                                    "Page References": "2",       "Explanation": "The SLM does not know this information. The retrieval of Page 2 is required."},
    {"Question": "How was the weather the night of the murder?",        "Category": "Traditional RAG",     "Correct Answer": "The weather was extremely hot.",                               "Page References": "26; 62", "Explanation": "The SLM does not know this information. The retrieval of Pages 26 and 62 is required. In page 26 Juror #10 says 'She's dying with the heat.' referring to the woman who seems to have seen the murder. In page 62 Juror #3 says 'It was a hot night, remember?'"},
    {"Question": "What does Juror #6 do for a living?",                 "Category": "Traditional RAG",     "Correct Answer": "He in a housepainter.",                                        "Page References": "1",       "Explanation": "The SLM does not know this information. The retrieval of Page 1 is required."},
    {"Question": "What is the charge against the boy in 12 Angry Men?", "Category": "Traditional RAG",     "Correct Answer": "Murder in the first degree which is a premeditated homicide.", "Page References": "4; 15",  "Explanation": "The SLM does not know this information. The retrieval of Pages 4 and 15 is required. In page 4 the Judge says 'you’ve heard a long and complex case. Murder in the firstdegree ... premeditated homicide ...is the most serious charge tried in our criminal courts.' In page 15 the same thing is mentioned again. Foreman says 'Just let’s remember we’ve got a first degree murder charge here.' "},

    # Reasoning / Graph RAG (5 questions)
    {"Question": "How was the woman's testimony proven inaccurate?",                                            "Category": "Reasoning/Graph RAG", "Correct Answer": "The woman claims she clearly saw the boy commit the crime, however it is revealed that she wears glasses. She was in bed at the time of the murder an thus it is unlikely she was wearing her glasses while sleeping. Therefore her identification of the boy is unreliable.",                               "Page References": "133; 141; 142; 143", "Explanation": "Requires entity extraction and multi-hop relationship reasoning. In page 133 Juror #4 says that according to her testimony she got a good look at the boy in the act of stabbing his father. In page 141 Juror #3 says that she had marks on her nose from glasses. In page 142 Juror #8 says that it’s logical to say that she wasn’t wearing them while she was in bed. In page 143 the same Juror says that he only knows that her eyesight is in question now, and thus her identification of the boy is unreliable."},
    {"Question": "Why does the train affect the ols man's testimony?",                                          "Category": "Reasoning/Graph RAG", "Correct Answer": "The old man claims that he has heard the boy threatening to kill his father. After that he seems to have heard the father's body hit the floor. However, it is shown that a loud train passed at that same time, which could have blocked the sound and therefore makes his testimony unreliable.",          "Page References": "62; 69; 70",         "Explanation": "Requires entity extraction and multi-hop relationship reasoning. In page 62 Juror #3 says that the old man who lived downstairs heard the kid yell cut 'I'm going to kill you'. A split second later the old man said that he have heard a body hit the floor. In page 69 Juror #8 says that he should have heard these noises while the el was roaring past his nose and thus it's not possible that he could have heard all these things. In page 70 Juror #8 says that if the old man had heard anything at all, he still couldn't have identified the voice with the el roaring by which makes his testimony unreliable."},
    {"Question": "How does the knife stop proving the boy's guilt?",                                            "Category": "Reasoning/Graph RAG", "Correct Answer": "Because Juror #8 proves that the knife is not unique, which means it cannot be used to directly link the boy to the crime.",                                                                                                                                                                                 "Page References": "39; 42; 43; 44",     "Explanation": "Requires entity extraction and multi-hop relationship reasoning. In page 39 Juror #4 says that the boy went to a neighborhood junk shop where he bought a switch-blade knife. He also said that the storekeeper who sold it to him identified it and said it was the only one of its kind he had ever had in stock. In page 42 it says that Juror #8 reaches into his pocket and swiftly withdraws a knife. In page 43 Juror #3 mentions that this knife is the same kind, of knife with the one the boy's father was murdered. In page 44 Juror #8 supports that it is possible for another person to have made the stabbing with the same kind of knife and consequently the knife cannot identify the boy."},
    {"Question": "Why does testing the old man’s walk impacts the verdict?",                                    "Category": "Reasoning/Graph RAG", "Correct Answer": "The old man in his testimony claims fast movement. However the Jurors recreate the walk and realise that it takes longer than claimed. Consequently the old man's testimony weakens and it can't influence the verdict that much.",                                                                          "Page References": "84; 88; 92",         "Explanation": "Requires entity extraction and multi-hop relationship reasoning. In page 84 Jurur #8 says that old man claims that he can get from his bed to his front door in fifteen seconds. In page 88 Juror #8 says that he wants to try this recreate this walk in order to see how long it took him. In page 92 it says that the duration of the recreated walk was thirty-three seconds exactly, a lot longer than claimed and the realisation that the testimony is doubtful becomes possible."},
    {"Question": "Which Juror's personal experiences makes him most likely to question the ols man's motives?", "Category": "Reasoning/Graph RAG", "Correct Answer": "Juror #9",                                                                                                                                                                                                                                                                                                   "Page References": "2; 62; 71",          "Explanation": "Requires entity extraction and multi-hop relationship reasoning. In page 2 Juror #9 is described as a gentle 70 years old man, long since defeated by life, and now merely waiting to die. In page 62 Juror #3 repeats one of the most important testimonies which comes from an old man. In page 71 Juror #9 says 'I think I know him better than anyone her. This is a quiet, frightened, insignificant old man who has been nothing all his life. A man like this needs to be recognized, to be listened to, to be quoted just once.' Juror #9 seems to understans the loneliness and desire for attention and applies this to the old man's situation. This makes Juror #9 question the old man's motives."},
]

df_questions = pd.DataFrame(questions)

# Insert TeamID as the first column
df_questions.insert(0, "TeamID", f"Team_{team_id}")

# Save to CSV
csv_filename = f"evaluation_dataset_team_{team_id}.csv"
df_questions.to_csv(csv_filename, index=False)
print(f"Saved {csv_filename}")
df_questions

Saved evaluation_dataset_team_28.csv


,TeamID,Question,Category,Correct Answer,Page References,Explanation
0,Team_28,How many days does a week have?,SLM only,A week has 7 days. This is a standard time per...,,It is general knowledge and doesn't require co...
1,Team_28,What is the chemical symbol for water?,SLM only,The chemical symbol for water is **H₂O**. This...,,It is general knowledge and doesn't require co...
2,Team_28,Which city is the capital of Greece?,SLM only,The capital of Greece is **Athens**.,,It is general knowledge and doesn't require co...
3,Team_28,Who painted the Mona Lisa?,SLM only,The Mona Lisa is a famous painting by **Leonar...,,It is general knowledge and doesn't require co...
4,Team_28,Which language is most widely known?,SLM only,The language most widely known is **English**....,,It is general knowledge and doesn't require co...
5,Team_28,How many kids does Juror #8 have?,Traditional RAG,He has three kids.,30,The SLM does not know this information. The re...
6,Team_28,Which Juror is retired?,Traditional RAG,Juror #9.,2,The SLM does not know this information. The re...
7,Team_28,How was the weather the night of the murder?,Traditional RAG,The weather was extremely hot.,26; 62,The SLM does not know this information. The re...
8,Team_28,What does Juror #6 do for a living?,Traditional RAG,He in a housepainter.,1,The SLM does not know this information. The re...
9,Team_28,What is the charge against the boy in 12 Angry...,Traditional RAG,Murder in the first degree which is a premedit...,4; 15,The SLM does not know this information. The re...
